[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raikar8/ML-Lab/blob/main/notebooks/01_triton_fusion.ipynb)

# 1 · Triton fundamentals: fused softmax and fused residual + RMSNorm
Week 15. Memory-bound ops are limited by bytes moved, not FLOPs. Here you'll measure that directly: a fused Triton softmax reads
and writes each element once, while the unfused PyTorch chain (max → subtract → exp → sum → divide) makes several passes over HBM.
The kernels are in `gpu/triton_kernels.py` (written from scratch; read them first).

In [ ]:
# Colab: Runtime → Change runtime type → GPU (a T4 is enough except where noted). Kaggle: Settings → Accelerator.
import os, sys
if not os.path.exists("ML-Lab") and not os.path.exists("gpu"):
    !git clone --depth 1 https://github.com/raikar8/ML-Lab.git
if os.path.exists("ML-Lab"):
    %cd ML-Lab
sys.path.insert(0, ".")
import torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Switch the runtime to a GPU"

In [ ]:
import triton.testing
def bench(fn):  # median milliseconds, with warm-up and L2 flushing handled by Triton
    return triton.testing.do_bench(fn, warmup=25, rep=100)

## Correctness first
Always check a kernel against a reference before timing it.

In [ ]:
from gpu import triton_kernels as tk
import torch.nn.functional as F
x = torch.randn(4096, 2048, device="cuda", dtype=torch.float16)
torch.testing.assert_close(tk.softmax(x), torch.softmax(x.float(), -1).half(), atol=2e-3, rtol=2e-3)
w, r = torch.rand(2048, device="cuda", dtype=torch.float16), torch.randn_like(x)
y, res = tk.add_rmsnorm(x, w, r)
torch.testing.assert_close(y, F.rms_norm((x + r).float(), (2048,), w.float(), eps=1e-6).half(), atol=2e-2, rtol=2e-2)
print("ok")

## Bandwidth: fused vs unfused
Achieved bandwidth = bytes that *must* move (read x once, write y once) ÷ time. Compare with your GPU's peak (T4 ≈ 320 GB/s,
L4 ≈ 300 GB/s, A100 ≈ 1.5–2 TB/s).

In [ ]:
def naive_softmax(x):
    m = x.max(-1, keepdim=True).values      # pass 1
    e = torch.exp(x - m)                     # pass 2 (+ write)
    return e / e.sum(-1, keepdim=True)       # passes 3–4

rows = []
for n in [256, 1024, 2048, 4096, 8192]:
    x = torch.randn(8192, n, device="cuda", dtype=torch.float16)
    gb = 2 * x.numel() * x.element_size() / 1e9
    t_tri, t_torch, t_naive = bench(lambda: tk.softmax(x)), bench(lambda: torch.softmax(x, -1)), bench(lambda: naive_softmax(x))
    rows.append((n, gb / t_tri * 1e3, gb / t_torch * 1e3, gb / t_naive * 1e3))
print(f"{'cols':>6} {'triton GB/s':>12} {'torch GB/s':>11} {'naive GB/s':>11}")
for r in rows:
    print(f"{r[0]:>6} {r[1]:>12.0f} {r[2]:>11.0f} {r[3]:>11.0f}")

In [ ]:
x = torch.randn(16384, 4096, device="cuda", dtype=torch.float16); r = torch.randn_like(x); w = torch.ones(4096, device="cuda", dtype=torch.float16)
eager = lambda: F.rms_norm(x + r, (4096,), w, eps=1e-6)
print(f"fused add+RMSNorm: {bench(lambda: tk.add_rmsnorm(x, w, r)):.3f} ms   eager: {bench(eager):.3f} ms   "
      f"compiled: {bench(torch.compile(eager)):.3f} ms")

## Your turn
1. Change the softmax kernel to process **several rows per program** (loop inside the kernel). When does it help?
2. Rows longer than one block (e.g. 65,536 columns) don't fit: write a two-pass (or online) version that loops over column blocks.
3. Explain in two sentences why `torch.compile` gets close to the hand-written kernel here — and when it wouldn't.